In [1]:
import glob
import os
import pandas as pd

# 1. Definir la ruta donde Slurm guardó los CSV individuales
output_dir = "/scratch/elena/9Li/results/isotopes_output"

# 2. Buscar todos los archivos de resumen que generó el cluster (summary_R*.csv)
csv_files = glob.glob(os.path.join(output_dir, "summary_R*.csv"))

print(f"Se han encontrado {len(csv_files)} archivos de resultados para fusionar.")

# 3. Leer todos los archivos y concatenarlos en un único DataFrame maestro
df_final = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# 4. Ordenar las filas por número de Run para que la tabla sea legible y esté organizada
df_final = df_final.sort_values(by="Run").reset_index(drop=True)

# 5. Guardar la tabla unificada por si quieres exportarla fuera del cluster en el futuro
master_csv_path = os.path.join(output_dir, "master_table_all_runs.csv")
df_final.to_csv(master_csv_path, index=False)
print(f"¡Éxito! Tabla maestra guardada en: {master_csv_path}")

# 6. Mostrar el DataFrame final formateado e interactivo de Jupyter
df_final

Se han encontrado 12 archivos de resultados para fusionar.
¡Éxito! Tabla maestra guardada en: /scratch/elena/9Li/results/isotopes_output/master_table_all_runs.csv


,Run,Beam p (MeV/c),N spills with pions,N pions (total),N 12B exp (20-50 ms),N Li9 exp (20-50 ms),N 16N exp (20-50 ms),N 12B exp (50-500 ms),N Li9 exp (50-500 ms),N 16N exp (50-500 ms)
0,1846,-340,165,260,5.20,9.51,1.03,3.02,63.44,14.95
1,1848,-340,165,260,4.21,8.60,1.03,2.45,57.43,14.90
2,1928,-260,132,157,0.76,4.21,0.62,0.44,28.09,8.93
3,1930,-260,68,80,0.48,2.54,0.32,0.28,16.93,4.60
4,1932,-260,183,227,1.99,6.56,0.89,1.16,43.78,12.95
5,1934,-260,102,121,0.87,3.45,0.48,0.51,22.99,6.93
6,1935,-260,115,138,1.37,4.29,0.55,0.80,28.60,7.93
7,1936,-260,105,125,0.16,2.94,0.49,0.09,19.59,7.10
8,1937,-260,147,186,1.74,5.36,0.73,1.02,35.80,10.65
9,1938,-260,70,85,0.44,1.99,0.33,0.26,13.27,4.84


In [6]:
df_final = df_final.sort_values(by="Run").reset_index(drop=True)

# ==============================================================================
# NUEVO: PASAR LAS ÚLTIMAS 6 COLUMNAS A VALOR POR SPILL
# ==============================================================================
# Buscamos de forma dinámica tus últimas 6 columnas que contienen "exp"
for col in df_final.columns:
    if "exp" in col:
        # Dividimos el total acumulado entre el número de spills con piones del run
        df_final[col] = (df_final[col] / df_final["N spills with pions"]).round(4)
        
        # Opcional: Renombramos la columna para dejar claro que ahora es "por spill"
        df_final = df_final.rename(columns={col: f"{col} per spill"})
# ==============================================================================

# 5. Guardar la tabla unificada por spill
master_csv_path = os.path.join(output_dir, "master_table_per_spill.csv")
df_final.to_csv(master_csv_path, index=False)
print(f"¡Éxito! Tabla maestra por spill guardada en: {master_csv_path}")

# 6. Mostrar el DataFrame final formateado e interactivo de Jupyter
df_final

¡Éxito! Tabla maestra por spill guardada en: /scratch/elena/9Li/results/isotopes_output/master_table_per_spill.csv


,Run,Beam p (MeV/c),N spills with pions,N pions (total),N 12B exp (20-50 ms) per spill,N Li9 exp (20-50 ms) per spill,N 16N exp (20-50 ms) per spill,N 12B exp (50-500 ms) per spill,N Li9 exp (50-500 ms) per spill,N 16N exp (50-500 ms) per spill
0,1846,-340,165,260,0.0315,0.0576,0.0062,0.0183,0.3845,0.0906
1,1848,-340,165,260,0.0255,0.0521,0.0062,0.0148,0.3481,0.0903
2,1928,-260,132,157,0.0058,0.0319,0.0047,0.0033,0.2128,0.0677
3,1930,-260,68,80,0.0071,0.0374,0.0047,0.0041,0.2490,0.0676
4,1932,-260,183,227,0.0109,0.0358,0.0049,0.0063,0.2392,0.0708
5,1934,-260,102,121,0.0085,0.0338,0.0047,0.0050,0.2254,0.0679
6,1935,-260,115,138,0.0119,0.0373,0.0048,0.0070,0.2487,0.0690
7,1936,-260,105,125,0.0015,0.0280,0.0047,0.0009,0.1866,0.0676
8,1937,-260,147,186,0.0118,0.0365,0.0050,0.0069,0.2435,0.0724
9,1938,-260,70,85,0.0063,0.0284,0.0047,0.0037,0.1896,0.0691


In [4]:
# 1. Definimos las reglas de agrupación sumando tus columnas exactas
aggregation_rules = {
    "N spills with pions": "sum",
    "N pions (total)": "sum"
}

# Buscamos de forma automática todas tus columnas de probabilidades "exp" para sumarlas
for col in df_final.columns:
    if "exp" in col:
        aggregation_rules[col] = "sum"

# 2. Agrupamos por el Beam momentum sumando absolutamente todas las filas
df_grouped = df_final.groupby("Beam p (MeV/c)", as_index=False).agg(aggregation_rules)

# 3. Mostramos la tabla final agrupada de 2 filas en tu Jupyter
df_grouped

,Beam p (MeV/c),N spills with pions,N pions (total),N 12B exp (20-50 ms) per spill,N Li9 exp (20-50 ms) per spill,N 16N exp (20-50 ms) per spill,N 12B exp (50-500 ms) per spill,N Li9 exp (50-500 ms) per spill,N 16N exp (50-500 ms) per spill
0,-340,330,520,0.057,0.1097,0.0124,0.0331,0.7326,0.1809
1,-260,1196,1452,0.080,0.3379,0.0479,0.0466,2.2536,0.6919
